---
title: Creating a SQLite database with parent and child tables
author: Aubrey Moore
date: 2023-02-09
---

# create_one_to_many_sql.ipynb

An example of setting up a SQLite3 database containing a one_to_many relationship between a parent and a child table.

Both tables have a unique id field for each record.

The biggest problem I had was setting up `try ... except` blocks to trap database integrity errors which caused crashes.

In [1]:
import sqlite3
import pandas as pd
# from icecream import ic
import os

# Functions

In [2]:
def db_is_empty(conn: sqlite3.Connection):
    """   
    """
    cursor = conn.cursor()
    sql = "SELECT COUNT(*) FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%';"
    cursor.execute(sql)
    num_tables = cursor.fetchone()[0]
    cursor.close()
    return num_tables == 0

In [3]:
def create_database(conn: sqlite3.Connection) -> None:
    """ 
    
    """
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS people (
            personid INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            age INTEGER
        );
    """)
    conn.commit()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            noteid INTEGER PRIMARY KEY AUTOINCREMENT,
            personid INTEGER,
            note TEXT,
            FOREIGN KEY(personid) REFERENCES people(personid) ON DELETE CASCADE  
        );
    """)
    conn.commit()
    cursor.close()

In [4]:
def populate_table_with_dataframe(conn: sqlite3.Connection, df: pd.DataFrame, db_path: str, table_name: str) -> None:
    """
    """
    try:
        df.to_sql(name=table_name, con=conn, if_exists='append', index=False)
        conn.commit()
    except sqlite3.IntegrityError as e:
        # Catch the specific error
        print(f"Caught an integrity error for user ID when populating people table: {e}")
        # Roll back the failed transaction
        conn.rollback()
        print("Transaction rolled back. Continuing with next record.")
    except Exception as e:
        # Catch any other potential errors
        print(f"An unexpected error occurred: {e}")
        print('recovering from error')
        conn.rollback()
    df = pd.read_sql(f'SELECT * FROM {table_name}', con=conn)

In [5]:
def get_rowcount(db_path: str, table_name: str, conn: sqlite3.Connection):
    cursor = conn.cursor()
    cursor.execute(f'SELECT COUNT(1) FROM {table_name};', conn)

# MAIN

In [6]:
# Initialize variables for this run

db_path = 'my_database.sqlite3'
delete_existing_database = True
people_data = [
    {'name': 'Aubrey Moore', 'age': 74},
    {'name': 'Jane Ginlo Moore', 'age': 72}
]
notes_data = [
    {'name': 'Aubrey Moore', 'note': 'an OK guy'},
    {'name': 'Jane Ginlo Moore', 'note': 'a beautiful lady with a beautiful voice'},
    {'name': 'Jane Ginlo Moore', 'note': 'She, who must be obeyed'}
]

In [7]:
# create database and populate tables

if delete_existing_database and os.path.exists(db_path):
    os.remove(db_path)
    
conn = sqlite3.connect(db_path)

if db_is_empty(conn):
    create_database(conn)

# populate 'people' table
df = pd.DataFrame(people_data)
populate_table_with_dataframe(conn, df=df, db_path=db_path, table_name='people')

# create a dict for mapping 'name' to 'personid'
cursor = conn.cursor()
cursor.execute('SELECT name, personid from people')
personid_map = dict(cursor.fetchall())
cursor.close()

# populate 'notes' table (df is reused)
df = pd.DataFrame(notes_data)

# replace 'name' column with 'personid' column by using 'name' as a key for the personid_map
df['personid'] = df['name'].map(personid_map)
df.drop('name', inplace=True, axis=1) # IMPORTANT

df = populate_table_with_dataframe(conn, df=df, db_path=db_path, table_name='notes')

In [8]:
# Load 'people' data into a dataframe

pd.read_sql('SELECT * FROM people', conn)

,personid,name,age
0,1,Aubrey Moore,74
1,2,Jane Ginlo Moore,72


In [9]:
# Load 'notes' table into a dafaframe

pd.read_sql('SELECT * FROM notes', conn)

,noteid,personid,note
0,1,1,an OK guy
1,2,2,a beautiful lady with a beautiful voice
2,3,2,"She, who must be obeyed"


In [10]:
# Close database connection

conn.close()
print('FINISHED')

FINISHED
